In [ ]:
# %load_ext cudf.pandas
import pandas as pd
import requests
from bs4 import BeautifulSoup

filename = "data/articles_batch_{}.csv"
failed_urls = "data/failed_urls_batch_{}.txt"

# MERGE

In [ ]:
merged_df = pd.DataFrame()
for i in range(0,3):
    temp_df = pd.read_csv(filename.format(i))
    merged_df = pd.concat([merged_df,temp_df],ignore_index=True)
    
success_count = merged_df.shape[0]

In [ ]:
failed_df = pd.DataFrame()
for i in range(0,3):
    temp_df = pd.read_csv(failed_urls.format(i),header=None)
    failed_df = pd.concat([failed_df,temp_df],ignore_index=True)
failed_count = failed_df.shape[0]

In [ ]:
with open("data/urls.txt", 'r') as f:
    all_urls = [line.strip() for line in f]

print(failed_count)

assert len(all_urls) == (success_count + failed_count)

## Null handling

In [ ]:
merged_df.info()

In [ ]:
null_categories = merged_df[merged_df['category'].isnull()]
null_contents = merged_df[merged_df['content'].isnull()]

In [ ]:
null_cat = null_categories.sample(1).iloc[0]['url']
null_cont = null_contents.sample(1).iloc[0]['url']
print(null_cat)
print(null_cont)

### Drop missing values

In [ ]:
merged_df = merged_df.dropna(subset=['category', 'content'])
merged_df.shape

In [ ]:
merged_df.head()

In [ ]:
import ast

def remove_hashtag(tags):
    # Convert string to list if needed
    if isinstance(tags, str):
        try:
            tags = ast.literal_eval(tags)
        except Exception:
            return []
    # Remove hashtags and strip whitespace
    return [tag.replace("#", "").strip() for tag in tags]


merged_df['time'] = pd.to_datetime(merged_df['time'], errors='coerce').dt.date
merged_df['content'] = merged_df['content'].apply(lambda x: x.strip())
merged_df['tags'] = merged_df['tags'].apply(remove_hashtag)

In [ ]:
merged_df.head()

In [ ]:
category_counts = merged_df['category'].value_counts()
categories = category_counts.index.to_numpy()
category_counts

In [ ]:
for cat in categories:
    sample = merged_df[merged_df['category'] == cat].sample(1).iloc[0]
    print(cat, sample['url'])

In [ ]:
merged_df.to_csv("data/baodautu.csv", index=False)